<a href="https://www.kaggle.com/code/jakomina/kickmix-v1?scriptVersionId=368305431" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Google Quantum AI — Verifiable Quantum Circuit Compilation · pipeline notebook

> **v2 (2026-11-09).** Section 10 now registers two real `invsqrt` builders (`invsqrt_v3`, fallback `invsqrt_v2`), built on the official `nr_reference_circuit.py`: trimmed register widths, 3 instead of 4 coefficient registers, and the leading-zero-pair count `s` kept in binary (4 qubits) with kickmix's quantum-shift `left_rotate`. A calibrated classical model (it reproduces the reference's 260 qubits, 7,735 Toffolis and cost 2,076,774.23 exactly) and an exhaustive simulation over all 3,133,452 x predict **v3: Q = 233, 6,935 Toffolis, cost ≈ 1,675,500 (ratio ≈ 1.24)** and v2: Q = 239, 7,327 Toffolis (≈ 1.15). `invsqrt` is verified in **full** official mode (minutes); `dqi`/`dfthcsos` stay in smoke mode, so only `invsqrt` can change in this version.

**Goal:** a reproducible, *never-regress* pipeline for the three benchmarks (`invsqrt`, `dqi`, `dfthcsos`). Score = geometric mean of `baseline_cost / cost`, the reference scores 1.0, and **one failing task rejects the whole submission** — so every change goes through a gate before it can reach `submission.zip`.

**What this notebook does**

| step | what | why it matters |
|---|---|---|
| 1–2 | find the competition files, load `kickmix` | no hard-coded layout; prints the tree so you can see exactly what shipped |
| 3 | exact rewrite passes on `.kmx` text (pair cancellation, phase folding, scratch-qubit reuse, id compaction) | identities on **all** basis inputs, so they can only lower cost |
| 4–5 | local fuzz harness (paper §II.C) + self-tests | proves the harness catches dirty scratch / wrong phase, and that the passes are exact (random-circuit differential test) |
| 6–8 | profile the reference, local cost proxies (paper formulas) | rank candidates quickly; the official score stays the authority |
| 7 | official verifiers | real CLI (`--salt`, JSON `status`/`score`), **calibrated on the reference** before it is trusted |
| 9–10 | optimise + plug-in builders | a candidate ships only if equivalent/verified **and** strictly cheaper |
| 11 | the metric's file-level checks, zip in the reference layout, manifest + SHA-256 | the metric rejects bad layouts before simulating; prize rules require reproducibility |

**How to run:** attach the competition data (and the *Verifiable Circuit Compilation Starter Kit* dataset for `dfthcsos`) → *Restart & Run All*. Read the output of cells 2 and 7, pin `OFFICIAL_CMD` if auto-detection did not, run again.

**Two-stage run (important):** the first run uses `CFG["OFFICIAL_MODE"] = "smoke"` — tiny shot counts, a few minutes. Cell 7 first *calibrates* each official verifier on the **reference** submission (it must PASS; if not, the raw command and output are printed and nothing may replace the reference). In smoke mode nothing is ever substituted. When calibration is green, set `CFG["OFFICIAL_MODE"] = "full"` for the real verification (hours; the dqi verifier needs ~6 GB per worker, see `DQI_WORKERS`). A candidate ships only if the official verifier PASSES it **and** its official cost (`score`, lower is better) beats the reference's under the same salt (dqi must win by > 0.5 %, its repetition factor depends on the salt).

**If cell 2 fails with `no usable kickmix` / `no attribute 'CircuitBuilder'`:** `kickmix` is *not* on PyPI and the published `qualtran` wheel (0.7.0) predates it, so `pip install` cannot fix it. Use the `kickmix*.so` shipped with the competition data (the notebook loads it by path), make sure no folder named `kickmix/` (e.g. a cloned Qualtran repo) is in the working directory or on `sys.path`, or — with Internet ON — set `CFG["BUILD_KICKMIX_IF_MISSING"] = True` to build it from Qualtran `main`.

**Honest limits:** the exact passes only remove what is redundant; real gains come from better circuits (builders, §10). The Python passes skip circuits longer than `MAX_OPS_PYTHON` (the DQI reference is tens of millions of gates). This notebook was self-tested against kickmix built from Qualtran `main`; it has **not** been run against the private-salt leaderboard verifiers.

## 1 · Configuration

In [1]:
# ── 1. Configuration, logging, determinism ───────────────────────────────────
import os, sys, re, json, time, glob, math, hashlib, shutil, subprocess, textwrap, random, zipfile, traceback, heapq
from pathlib import Path
from fractions import Fraction
from decimal import Decimal, localcontext
import numpy as np

T0 = time.time()
ON_KAGGLE = Path("/kaggle/input").is_dir()

CFG = dict(
    # where the competition files may live (first existing directory wins; otherwise we search /kaggle/input)
    COMP_CANDIDATES=[
        os.environ.get("QUCO_COMP", ""),
        "/kaggle/input/competitions/google-verifiable-quantum-circuit-compilation",
        "/kaggle/input/google-verifiable-quantum-circuit-compilation",
    ],
    WORK=os.environ.get("QUCO_WORK", "/kaggle/working" if ON_KAGGLE else "/tmp/quco_work"),
    SEED=20261025,
    TIME_BUDGET_S=6 * 3600,        # whole notebook; passes check time_left() before starting heavy work
    RUN_OPTIMIZER=True,            # exact rewrite passes on the reference circuits
    RUN_OFFICIAL=True,             # call the competition verifiers when they can be located
    ALLOW_UNVERIFIED_SUBSTITUTION=False,   # NEVER ship a changed circuit the official verifier did not pass
    MAX_OPS_PYTHON=3_000_000,      # circuits longer than this skip the Python rewrite passes
    FUZZ_BATCH=512,
    INVSQRT_SAMPLE=0,              # 0 = exhaustive: all 3,133,452 x (about a minute per circuit)
    BUILD_KICKMIX_IF_MISSING=False,   # set True (and turn Internet ON) to build kickmix from Qualtran main
    # --- official verifiers (CLI taken from how the public reproducibility repo drives them) ---
    OFFICIAL_MODE="smoke",         # "smoke" = tiny shot counts (plumbing check, minutes) | "full" = the real thing (hours)
    OFFICIAL_MODE_BY_TASK=dict(invsqrt="full", dqi="smoke", dfthcsos="smoke"),   # per task; invsqrt full takes minutes
    OFFICIAL_SALT="quco",          # same salt for reference and candidate so the comparison is apples-to-apples
    DQI_WORKERS=2,                 # each dqi verifier worker needs ~6 GB of RAM
    OFFICIAL_TIMEOUT_S=3 * 3600,
    ALLOW_SMOKE_SUBSTITUTION=False,  # a smoke-mode pass NEVER replaces the reference unless you set this
    MIN_REL_GAIN=dict(invsqrt=0.0, dqi=0.005, dfthcsos=0.0),   # dqi's R depends on the salt (about +-0.5 %), so demand more
    SMOKE_ARGS=dict(invsqrt=["--max-x", "20480", "--shots-per-x", "1"],   # num_shots must be a multiple of the batch size
                    dqi=["--num-shots", "2560"],                          # 256 shots would give eps+ > 1/16 (no finite score)
                    dfthcsos=["--shots-per-address", "1"]),
    BETA=1,                        # invsqrt phase = exp(2*pi*i*BETA/sqrt(x))
)
WORK = Path(CFG["WORK"]); WORK.mkdir(parents=True, exist_ok=True)
(WORK / "candidates").mkdir(exist_ok=True)
random.seed(CFG["SEED"]); np.random.seed(CFG["SEED"] % 2**32)

def elapsed(): return time.time() - T0
def time_left(): return CFG["TIME_BUDGET_S"] - elapsed()
def log(*a):
    print(f"[{elapsed():8.1f}s]", *a, flush=True)

LEDGER = []   # every candidate we evaluate, written to ledger.json at the end
def ledger(**row):
    row["t"] = round(elapsed(), 1); LEDGER.append(row)
    (WORK / "ledger.json").write_text(json.dumps(LEDGER, indent=1, default=str))

def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""): h.update(chunk)
    return h.hexdigest()

log("python", sys.version.split()[0], "| numpy", np.__version__, "| on_kaggle", ON_KAGGLE, "| work", WORK)


[     0.0s] python 3.13.18 | numpy 2.1.3 | on_kaggle True | work /kaggle/working


## 2 · Discover files & load kickmix

In [2]:
# ── 2. Discover the competition files and load kickmix ───────────────────────
def find_comp_dir():
    for p in CFG["COMP_CANDIDATES"]:
        if p and Path(p).is_dir(): return Path(p)
    root = Path("/kaggle/input")
    if root.is_dir():                                    # fall back to a shallow search
        for dp, dns, fns in os.walk(root):
            if dp[len(str(root)):].count(os.sep) > 4: dns[:] = []; continue
            if any(f.endswith(".kmb") or f.startswith("kickmix") and f.endswith(".so") for f in fns):
                return Path(dp)
    return None

def show_tree(root, max_depth=4, per_dir=40):
    root = Path(root)
    for dp, dns, fns in os.walk(root):
        depth = Path(dp).relative_to(root).parts.__len__()
        if depth > max_depth: dns[:] = []; continue
        dns.sort(); fns.sort()
        ind = "  " * depth
        print(f"{ind}{Path(dp).name or dp}/")
        for f in fns[:per_dir]:
            try: sz = (Path(dp) / f).stat().st_size
            except OSError: sz = -1
            print(f"{ind}  {f}  ({sz:,} B)")
        if len(fns) > per_dir: print(f"{ind}  … {len(fns) - per_dir} more files")

COMP = find_comp_dir()
log("competition dir:", COMP)
if COMP is not None: show_tree(COMP)
INPUTS = sorted(p.name for p in Path("/kaggle/input").iterdir()) if ON_KAGGLE else []
log("other /kaggle/input entries:", INPUTS)

# ── kickmix loading ──────────────────────────────────────────────────────────────────────────────
# kickmix is NOT on PyPI and the published `qualtran` wheel (0.7.0) predates it, so `pip install` cannot help.
# Valid sources, in the order tried:  (1) the kickmix*.so shipped with the competition data (built for the Kaggle
# image), loaded BY PATH;  (2) an already-installed build;  (3) a source build from Qualtran `main` (needs Internet ON).
# A folder called `kickmix/` (e.g. the Qualtran repo's project directory on sys.path or in the working directory) is a
# namespace package with no attributes: `import kickmix` then "works" but `km.CircuitBuilder` raises AttributeError.
import importlib.util, importlib
KM_REQUIRED = {"Circuit": ["from_file", "to_file", "num_qubits", "num_bits", "register_data", "operation_statistics", "reaction_depth"],
               "Simulator": ["use_same_registers_as", "write_across_shots", "read_across_shots", "read_shot_phase", "clear_for_shot", "do"]}

def _km_problems(mod):
    """[] when mod is a usable kickmix; otherwise a list of human-readable problems."""
    bad = []
    for cls, attrs in KM_REQUIRED.items():
        c = getattr(mod, cls, None)
        if c is None: bad.append(f"module has no class {cls!r}"); continue
        bad += [f"{cls}.{a} missing" for a in attrs if not hasattr(c, a)]
    return bad

def _describe(mod):
    return (f"{type(mod).__name__} file={getattr(mod, '__file__', None)} path={list(getattr(mod, '__path__', []))} "
            f"public names={[n for n in dir(mod) if not n.startswith('_')][:12]}")

def _load_so(so):
    sys.modules.pop("kickmix", None)
    spec = importlib.util.spec_from_file_location("kickmix", so)
    mod = importlib.util.module_from_spec(spec); sys.modules["kickmix"] = mod
    spec.loader.exec_module(mod); return mod

def build_kickmix_from_source(jobs=4, dest=None):
    """Last resort (Internet ON, g++ available): build kickmix from Qualtran main. ~5-15 min depending on cores."""
    dest = Path(dest or WORK / "qualtran_src")
    if not (dest / "kickmix").is_dir():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/quantumlib/Qualtran.git", str(dest)], check=True)
    try:
        import pybind11, setuptools                                            # already there -> skip pip entirely
    except ImportError:
        pip = [sys.executable, "-m", "pip", "install", "-q", "pybind11", "setuptools", "wheel"]
        if subprocess.run(pip).returncode: subprocess.run(pip + ["--break-system-packages"], check=True)
    subprocess.run([sys.executable, "setup.py", "build_ext", "--inplace", f"-j{jobs}"], cwd=dest / "kickmix", check=True)
    so = sorted((dest / "kickmix").glob("kickmix*.so"))
    if not so: raise RuntimeError("build finished but no kickmix*.so was produced")
    return _load_so(so[0])

def load_kickmix():
    notes = []
    roots = [r for r in (COMP, Path("/kaggle/input")) if r and Path(r).is_dir()]
    seen = set()
    for r in roots:                                                            # (1) the .so that ships with the data
        for so in sorted(Path(r).rglob("kickmix*.so")):
            if so in seen: continue
            seen.add(so)
            try:
                mod = _load_so(so); bad = _km_problems(mod)
                if not bad: return mod
                notes.append(f"{so}: loads but is unusable ({'; '.join(bad[:4])})")
            except Exception as e:
                notes.append(f"{so}: {e!r}")
    sys.modules.pop("kickmix", None)                                           # (2) an installed build, if it is the real thing
    try:
        mod = importlib.import_module("kickmix"); bad = _km_problems(mod)
        if not bad: return mod
        notes.append("`import kickmix` returned " + _describe(mod) + " -> a FOLDER named 'kickmix' shadows the compiled module "
                     "(remove it from sys.path / the working directory) or it is a different build")
    except ImportError as e:
        notes.append(f"`import kickmix`: {e!r}")
    if CFG.get("BUILD_KICKMIX_IF_MISSING"):                                    # (3) opt-in source build
        log("building kickmix from source …")
        mod = build_kickmix_from_source(); bad = _km_problems(mod)
        if not bad: return mod
        notes.append("source build is unusable: " + "; ".join(bad[:4]))
    raise ImportError("no usable kickmix.\n  - " + "\n  - ".join(notes) +
                      "\nFix: attach the competition data (it ships kickmix.so), or turn Internet ON and set "
                      "CFG['BUILD_KICKMIX_IF_MISSING'] = True, then re-run this cell.")

km = load_kickmix()
log("kickmix loaded from", getattr(km, "__file__", "?"), "| public names:", [n for n in dir(km) if not n.startswith("_")])
log("API check:", "complete" if not _km_problems(km) else _km_problems(km))


[     0.5s] competition dir: /kaggle/input/competitions/google-verifiable-quantum-circuit-compilation
google-verifiable-quantum-circuit-compilation/
  verifiable_circuit_compilation.pdf  (1,092,999 B)
  kickmix/
    LICENSE.txt  (11,358 B)
    kickmix.so  (3,220,392 B)
  sample_submission/
    dqi.kmb  (886,118,180 B)
    invsqrt.kmb  (579,721 B)
    dfthcsos/
      angle_k_0.kmb  (1,671,932 B)
      angle_k_1.kmb  (1,662,896 B)
      angle_k_2.kmb  (1,665,821 B)
      angle_k_3.kmb  (1,666,802 B)
      angle_k_4.kmb  (1,954,082 B)
      angle_k_5.kmb  (1,948,583 B)
      angle_k_6.kmb  (1,947,980 B)
      angle_k_7.kmb  (1,968,527 B)
      angle_k_8.kmb  (1,957,124 B)
      angle_k_9.kmb  (1,953,830 B)
      submission.npz  (4,537,436 B)
  verifiers/
    LICENSE.txt  (11,358 B)
    dfthcsos/
      FeMoCo76o113e.fcidump  (128,128,348 B)
      FeMoCo76o113e.fixed_shifts.npz  (61,028 B)
      SCORING_README.md  (22,477 B)
      __init__.py  (326 B)
      verify.py  (31,752 B)
      examp

## 3 · Exact rewrite passes

In [3]:
# ── 3. Exact circuit rewrite passes on .kmx text ─────────────────────────────
# Every pass below is an identity on ALL basis inputs (not just the promise region):
#   cancel_pass   : A … A  →  nothing, for self-inverse gates with nothing in between on their wires
#   fold_phases   : Z_POW(a) … Z_POW(b) on one qubit, only commuting (control/diagonal) uses between → Z_POW(a+b)
#   reuse_scratch : a scratch qubit whose last op is an unconditional HMR/R is |0> afterwards, so a later scratch
#                   qubit (which starts in |0> anyway) can take over its id  →  fewer qubits
#   compact_scratch: fill id gaps below the register qubits (register qubit ids never change)
# HMR/R order is untouched, so a seeded simulator draws the same random bits before and after.

META = {"REGISTER", "APPEND_TO_REGISTER", "INFO_TAG"}
BARRIER = {"PUSH_CONDITION", "POP_CONDITION"}
SELF_INV = {"X", "CX", "CCX", "Z", "CZ", "CCZ", "SWAP", "CSWAP"}
_QUOTED = re.compile(r'"[^"]*"')
_WIRE = re.compile(r"^[qb]\d+$")

class Op:
    __slots__ = ("name", "t", "cond", "quoted", "dead")
    def __init__(self, name, t, cond=None, quoted=None):
        self.name, self.t, self.cond, self.quoted, self.dead = name, t, cond, quoted, False
    def text(self):
        parts = [self.name] + self.t
        if self.cond: parts += ["if", self.cond]
        if self.quoted: parts.append(self.quoted)
        return " ".join(parts)

def parse_kmx(text):
    ops = []
    for line in text.split("\n"):
        line = line.split("#", 1)[0].strip()           # '#' cannot occur inside quoted text (it is escaped as \P)
        if not line: continue
        quoted = None
        m = _QUOTED.search(line)
        if m: quoted = m.group(0); line = (line[:m.start()] + line[m.end():]).strip()
        tok = line.split()
        cond = None
        if "if" in tok:
            k = tok.index("if"); cond = tok[k + 1]; tok = tok[:k] + tok[k + 2:]
        ops.append(Op(tok[0], tok[1:], cond, quoted))
    return ops

def dump_kmx(ops): return "\n".join(o.text() for o in ops if not o.dead) + "\n"

def roles(op):
    """[(wire, role)]: N = non-diagonal use of a qubit, D = diagonal/control use, W = bit write, C = condition read"""
    n, t = op.name, op.t
    r = []
    if   n == "X":   r = [(t[0], "N")]
    elif n == "CX":  r = [(t[0], "D"), (t[1], "N")]
    elif n == "CCX": r = [(t[0], "D"), (t[1], "D"), (t[2], "N")]
    elif n in ("Z", "CZ", "CCZ"): r = [(w, "D") for w in t]
    elif n == "Z_POW": r = [(t[0], "D")]
    elif n == "SWAP":  r = [(t[0], "N"), (t[1], "N")]
    elif n == "CSWAP": r = [(t[0], "D"), (t[1], "N"), (t[2], "N")]
    elif n == "R":     r = [(t[0], "N")]
    elif n == "HMR":   r = [(t[0], "N"), (t[1], "W")]
    elif n in ("BIT_STORE0", "BIT_STORE1", "BIT_INVERT"): r = [(t[0], "W")]
    elif n == "DEBUG_PRINT": r = [(w, "N") for w in t if _WIRE.match(w)]
    if op.cond: r.append((op.cond, "C"))
    return r

def wires(op): return [w for w, _ in roles(op)]

def _key(op):
    n, t = op.name, list(op.t)
    if n in ("CZ", "CCZ", "SWAP"): t = sorted(t)
    elif n == "CCX":   t = sorted(t[:2]) + t[2:]
    elif n == "CSWAP": t = t[:1] + sorted(t[1:])
    return (n, tuple(t), op.cond)

def cancel_pass(ops):
    hist, removed = {}, 0
    for i, op in enumerate(ops):
        if op.dead or op.name in META: continue
        if op.name in BARRIER: hist.clear(); continue
        ws = set(wires(op))
        if op.name in SELF_INV and ws:
            last = {(hist.get(w) or [None])[-1] for w in ws}
            if len(last) == 1 and None not in last:
                j = next(iter(last))
                if ops[j].name == op.name and _key(ops[j]) == _key(op):
                    ops[j].dead = op.dead = True
                    for w in ws: hist[w].pop()
                    removed += 2; continue
        for w in ws: hist.setdefault(w, []).append(i)
    return removed

def _fmt_exp(e):
    e = e % 2
    if e.denominator & (e.denominator - 1): raise ValueError("exponent is not a dyadic rational")
    with localcontext() as ctx:
        ctx.prec = 400
        s = format(Decimal(e.numerator) / Decimal(e.denominator), "f")
    return s if "." in s else s + ".0"

def fold_phases(ops):
    pending, ver, merged = {}, {}, 0
    for i, op in enumerate(ops):
        if op.dead or op.name in META: continue
        if op.name in BARRIER: pending.clear(); continue
        if op.name == "Z_POW":
            q = op.t[0]
            key = (op.cond, ver.get(op.cond, 0) if op.cond else 0)
            d = pending.setdefault(q, {})
            if key in d:
                j = d[key]
                try: ops[j].t[1] = _fmt_exp(Fraction(ops[j].t[1]) + Fraction(op.t[1]))
                except ValueError: d[key] = i; continue
                op.dead = True; merged += 1
            else:
                d[key] = i
            continue
        for w, role in roles(op):
            if role == "N" and w[0] == "q": pending.pop(w, None)
            elif role == "W": ver[w] = ver.get(w, 0) + 1
    dropped = 0
    for op in ops:                                     # exponent 0 (mod 2) is the identity
        if not op.dead and op.name == "Z_POW":
            try: z = Fraction(op.t[1]) % 2 == 0
            except ValueError: z = False
            if z: op.dead = True; dropped += 1
    return merged, dropped

def _rename(ops, mp):
    for op in ops:
        if op.dead: continue
        op.t = [mp.get(w, w) if _WIRE.match(w) else w for w in op.t]

def reuse_scratch(ops):
    reg_q, depth, first, last, clean = set(), 0, {}, {}, {}
    for i, op in enumerate(ops):
        if op.dead: continue
        if op.name == "APPEND_TO_REGISTER": reg_q |= {w for w in op.t if w[0] == "q"}; continue
        if op.name in META: continue
        if op.name == "PUSH_CONDITION": depth += 1; continue
        if op.name == "POP_CONDITION":  depth -= 1; continue
        for w, _ in roles(op):
            if w[0] != "q": continue
            first.setdefault(w, i); last[w] = i
            clean[w] = op.name in ("R", "HMR") and op.cond is None and depth == 0 and op.t[0] == w
    mapping, free = {}, []                             # free: heap of (last_index, physical_qubit) that ended clean
    nq_before = len({w for w in first})
    for q in sorted((w for w in first if w not in reg_q), key=lambda w: first[w]):
        if free and free[0][0] < first[q]:
            _, p = heapq.heappop(free)
        else:
            p = q
        mapping[q] = p
        if clean[q]: heapq.heappush(free, (last[q], p))
    _rename(ops, mapping)
    return nq_before - len({mapping.get(w, w) for w in first})

def compact_scratch(ops):
    reg_q, used = set(), set()
    for op in ops:
        if op.dead: continue
        if op.name == "APPEND_TO_REGISTER": reg_q |= {w for w in op.t if w[0] == "q"}
        elif op.name not in META: used |= {w for w in wires(op) if w[0] == "q"}
    scratch = sorted((w for w in used if w not in reg_q), key=lambda w: int(w[1:]))
    taken = {int(w[1:]) for w in reg_q}
    mapping, nxt = {}, 0
    for w in scratch:
        while nxt in taken: nxt += 1
        mapping[w] = f"q{nxt}"; nxt += 1
    _rename(ops, mapping)
    return len(scratch)

def optimize_text(text, rounds=3):
    ops = parse_kmx(text)
    rep = dict(ops_in=len(ops), cancelled=0, phase_merged=0, phase_dropped=0, qubits_reused=0)
    for _ in range(rounds):
        c = cancel_pass(ops); m, d = fold_phases(ops)
        rep["cancelled"] += c; rep["phase_merged"] += m; rep["phase_dropped"] += d
        if c + m + d == 0: break
    rep["qubits_reused"] = reuse_scratch(ops)
    compact_scratch(ops)
    rep["ops_out"] = sum(not o.dead for o in ops)
    return dump_kmx(ops), rep

def optimize_circuit(circ):
    n = len(circ)
    if n > CFG["MAX_OPS_PYTHON"]:
        return None, dict(skipped=f"{n:,} instructions > MAX_OPS_PYTHON")
    text, rep = optimize_text(str(circ))
    return km.Circuit(text), rep

def circuit_report(c):
    st = c.operation_statistics()
    regs = {str(k): len(v) for k, v in c.register_data.items() if isinstance(k, str)}
    return dict(qubits=c.num_qubits, bits=c.num_bits, instructions=len(c),
                toffoli_like=st.num_toffoli_like, t_like=st.num_t_like,
                z_pow_fine=st.num_z_pow(min_clifford_hierarchy_level=4),
                reaction_depth=c.reaction_depth(), registers=regs)


## 4 · Local fuzz harness

In [4]:
# ── 4. Local verification harness (paper §II.C: classical fuzz testing of kickmix circuits) ──
def _id_of(x):
    """km.q(3) / km.b(3) -> 3 (works whatever the binding exposes)"""
    for attr in ("id", "index", "value"):
        v = getattr(x, attr, None)
        if isinstance(v, int): return v
    m = re.search(r"(\d+)", repr(x)); return int(m.group(1))

def bit_matrix(vals, n):
    vals = np.asarray(vals, dtype=np.uint64)
    return ((vals[None, :] >> np.arange(n, dtype=np.uint64)[:, None]) & np.uint64(1)).astype(bool)

def reg_ids(circ, name):
    return [_id_of(v) for v in circ.register_data[name]]

def check_phase_oracle(circ, reg, xs, target_units, tol_units, batch=None, seed=1):
    """Run |x>|0> -> e^{2 pi i f(x)}|x>|0> on the classical inputs xs (one shot each).
    Phase is compared in units of 2^-64 turns.  Returns a dict of failure counts and the worst error/tolerance ratio."""
    batch = batch or CFG["FUZZ_BATCH"]
    xs = np.asarray(xs, dtype=np.uint64)
    n = len(circ.register_data[reg]); xid = set(reg_ids(circ, reg))
    scratch = [km.q(i) for i in range(circ.num_qubits) if i not in xid]
    sim = km.Simulator(batch_size=batch, debug_print=False, seed=seed)
    sim.use_same_registers_as(circ)
    bad_x = bad_scratch = bad_phase = 0; worst = 0.0; first_bad = None
    M = 1 << 64
    for s in range(0, len(xs), batch):
        chunk = xs[s:s + batch]; m = len(chunk)
        pad = np.zeros(batch, dtype=np.uint64); pad[:m] = chunk
        sim.clear_for_shot()
        sim.write_across_shots(reg, bit_matrix(pad, n))
        sim.do(circ)
        back = sim.read_across_shots(reg)
        x_ok = (back == bit_matrix(pad, n)).all(axis=0)[:m]
        dirty = sim.read_across_shots(scratch).any(axis=0)[:m] if scratch else np.zeros(m, bool)
        for k in range(m):
            x = int(chunk[k])
            ph = Fraction(sim.read_shot_phase(k)) / 2 % 1          # half turns -> turns in [0,1)
            got = int(ph * M)
            d = (got - target_units(x)) % M; d = min(d, M - d)
            tol = tol_units(x)
            worst = max(worst, d / tol)
            ok_ph = d <= tol
            bad_x += not x_ok[k]; bad_scratch += bool(dirty[k]); bad_phase += not ok_ph
            if first_bad is None and not (x_ok[k] and not dirty[k] and ok_ph): first_bad = x
    return dict(shots=len(xs), bad_register=int(bad_x), bad_scratch=int(bad_scratch), bad_phase=int(bad_phase),
                worst_err_over_tol=worst, first_bad_x=first_bad,
                passed=(bad_x == 0 and bad_scratch == 0 and bad_phase == 0))

def invsqrt_targets(beta=1):
    """target phase (turns, units 2^-64) of exp(2 pi i beta / sqrt(x)), and the tolerance 2^-24 beta/sqrt(x) turns"""
    def target(x): return math.isqrt(((1 << 128) * beta * beta) // x) % (1 << 64)
    def tol(x):    return max(1, math.isqrt(((1 << 80) * beta * beta) // x))
    return target, tol

def random_inputs(rng, nbits, count):
    hi = (1 << nbits) - 1
    return np.array([rng.randint(1, hi) for _ in range(count)], dtype=np.uint64)

def equiv_check(a, b, batch=128, rounds=3, seed=7):
    """Circuit-vs-circuit: same random inputs on every named register, same PRNG seed.
    Compares register contents, 'any scratch dirty', every classical bit and the tracked phase of each shot."""
    rng = np.random.default_rng(seed)
    regs = {k: v for k, v in a.register_data.items() if isinstance(k, str)}
    reg_qs = {_id_of(v) for vs in regs.values() for v in vs if repr(v).startswith("km.q")}
    scr_a = [km.q(i) for i in range(a.num_qubits) if i not in reg_qs]
    scr_b = [km.q(i) for i in range(b.num_qubits) if i not in reg_qs]
    bits_a = [km.b(i) for i in range(a.num_bits)]; bits_b = [km.b(i) for i in range(b.num_bits)]
    for r in range(rounds):
        sims = []
        for c in (a, b):
            s = km.Simulator(batch_size=batch, debug_print=False, seed=seed + r)
            s.use_same_registers_as(c); sims.append(s)
        for name, vs in regs.items():
            m = rng.integers(0, 2, size=(len(vs), batch)).astype(bool)
            for s in sims: s.write_across_shots(name, m)
        for s, c in zip(sims, (a, b)): s.do(c)
        for name in regs:
            if not (sims[0].read_across_shots(name) == sims[1].read_across_shots(name)).all(): return False, f"register {name!r} differs"
        da = sims[0].read_across_shots(scr_a).any(axis=0) if scr_a else np.zeros(batch, bool)
        db = sims[1].read_across_shots(scr_b).any(axis=0) if scr_b else np.zeros(batch, bool)
        if not (da == db).all(): return False, "scratch dirtiness differs"
        if bits_a and bits_b and not (sims[0].read_across_shots(bits_a) == sims[1].read_across_shots(bits_b)).all(): return False, "classical bits differ"
        for k in range(batch):
            if sims[0].read_shot_phase(k) != sims[1].read_shot_phase(k): return False, f"phase differs in shot {k}"
    return True, "identical on all compared quantities"


## 5 · Self-tests

In [5]:
# ── 5. Self-tests: the harness and every rewrite pass must be right before we trust them on real circuits ──
def toy_phase_circuit(nbits=6, broken=None):
    """phase = x / 2**nbits turns, written as kmx text (no CircuitBuilder, so it works with any kickmix build)"""
    L = [f"APPEND_TO_REGISTER q{i} r0" for i in range(nbits)] + ['REGISTER r0 "x"']
    for k in range(nbits):                                   # exponent unit is the half turn
        L.append(f"Z_POW q{k} {_fmt_exp(Fraction(1 << k, 1 << (nbits - 1)))}")
    if broken == "dirty": L.append(f"X q{nbits}")
    if broken == "phase": L.append(f"Z_POW q0 {_fmt_exp(Fraction(1, 1 << 40))}")
    return km.Circuit("\n".join(L) + "\n")

def _selftest_harness():
    nb = 6
    xs = np.arange(0, 1 << nb, dtype=np.uint64)
    target = lambda x: (x * (1 << 64) // (1 << nb)) % (1 << 64)         # x / 2^nb turns
    tol = lambda x: 4
    ok = check_phase_oracle(toy_phase_circuit(nb), "x", xs, target, tol)
    d1 = check_phase_oracle(toy_phase_circuit(nb, "dirty"), "x", xs, target, tol)
    d2 = check_phase_oracle(toy_phase_circuit(nb, "phase"), "x", xs, target, lambda x: 1)
    assert ok["passed"], ok
    assert not d1["passed"] and d1["bad_scratch"] > 0, d1
    assert not d2["passed"] and d2["bad_phase"] > 0, d2
    log("harness: correct circuit passes; dirty-scratch and wrong-phase circuits are rejected")

def random_kmx(rng, nq=14, nb=6, nops=140):
    """valid kickmix circuit with registers a (qubits), c (bits), scratch lifecycles and many cancellable/mergeable patterns"""
    L = ["APPEND_TO_REGISTER q0 r0", "APPEND_TO_REGISTER q1 r0", "APPEND_TO_REGISTER q2 r0", 'REGISTER r0 "a"',
         "APPEND_TO_REGISTER b0 r1", "APPEND_TO_REGISTER b1 r1", 'REGISTER r1 "c"']
    ng = 6                                                  # qubits 0..5: general use; qubits 6.. : scratch lifecycles only
    Q = lambda: f"q{rng.randrange(ng)}"; B = lambda: f"b{rng.randrange(nb)}"
    def q3():
        s = rng.sample(range(ng), 3); return [f"q{i}" for i in s]
    def exp(): return f"{rng.randrange(0, 200) / 64}"
    depth = 0
    def one():
        k = rng.random(); c = f" if {B()}" if rng.random() < 0.2 else ""
        if k < .10: return f"X {Q()}{c}"
        if k < .22: a, b_, _ = q3(); return f"CX {a} {b_}{c}"
        if k < .30: a, b_, d = q3(); return f"CCX {a} {b_} {d}{c}"
        if k < .36: return f"Z {Q()}{c}"
        if k < .42: a, b_, _ = q3(); return f"CZ {a} {b_}{c}"
        if k < .46: a, b_, d = q3(); return f"CCZ {a} {b_} {d}{c}"
        if k < .62: return f"Z_POW {Q()} {exp()}{c}"
        if k < .67: a, b_, _ = q3(); return f"SWAP {a} {b_}{c}"
        if k < .71: a, b_, d = q3(); return f"CSWAP {a} {b_} {d}{c}"
        if k < .74: return f"NEG{c}"
        if k < .78: return f"HMR {Q()} {B()}{c}"
        if k < .80: return f"R {Q()}{c}"
        if k < .83: return f"BIT_INVERT {B()}"
        if k < .85: return f"BIT_STORE{rng.randrange(2)} {B()}"
        return f"X {Q()}"
    for _ in range(nops):
        r = rng.random()
        if r < 0.15:                                        # an op, optionally something unrelated, then the same op again
            op = one()
            if op.startswith(("HMR", "R ", "NEG", "BIT")): L.append(op); continue
            L.append(op)
            if rng.random() < .5: L.append(one())
            L.append(op)
        elif r < 0.22:                                      # scratch lifecycle ending in an unconditional HMR
            s = f"q{rng.randrange(ng, nq)}"
            L += [f"R {s}", f"CX q{rng.randrange(3)} {s}", f"CCX q{rng.randrange(3)} q{rng.randrange(3, ng)} {s}", f"HMR {s} b{rng.randrange(2, nb)}"]
        elif r < 0.26 and depth == 0:
            L.append(f"PUSH_CONDITION if {B()}"); depth = 1
        elif r < 0.34 and depth == 1:
            L.append("POP_CONDITION"); depth = 0
        else:
            L.append(one())
    if depth: L.append("POP_CONDITION")
    return "\n".join(L) + "\n"

def _selftest_passes(n_circuits=150):
    rng = random.Random(CFG["SEED"]); tot = dict(cancelled=0, phase_merged=0, phase_dropped=0, qubits_reused=0)
    for i in range(n_circuits):
        text = random_kmx(rng)
        a = km.Circuit(text)
        assert km.Circuit(str(a)) == a                      # canonical text round trip
        new_text, rep = optimize_text(str(a))
        b = km.Circuit(new_text)
        ok, why = equiv_check(a, b, batch=64, rounds=2, seed=100 + i)
        assert ok, f"pass changed behaviour (circuit {i}): {why}\n--- original\n{text}\n--- optimized\n{new_text}"
        for k in tot: tot[k] += rep[k]
    assert tot["cancelled"] and tot["phase_merged"] and tot["qubits_reused"], tot
    log(f"passes: {n_circuits} random circuits, optimized == original on registers, bits, scratch and phase |", tot)

SELFTEST_OK = False
try:
    _selftest_harness()
    _selftest_passes()
    SELFTEST_OK = True
    log("self-tests passed")
except Exception as e:     # an API difference or a real bug: do NOT trust the rewrite passes, but keep the notebook alive
    log("SELF-TESTS FAILED -> the rewrite passes are disabled this run:", repr(e)); traceback.print_exc()


[     0.7s] harness: correct circuit passes; dirty-scratch and wrong-phase circuits are rejected
[     1.3s] passes: 150 random circuits, optimized == original on registers, bits, scratch and phase | {'cancelled': 4222, 'phase_merged': 829, 'phase_dropped': 37, 'qubits_reused': 347}
[     1.3s] self-tests passed


## 6 · Reference submission

In [6]:
# ── 6. Load the reference submission (scores 1.0) and profile it ─────────────
TASKS = ("invsqrt", "dqi", "dfthcsos")

def find_reference(comp):
    """The reference submission = the directory holding invsqrt.kmb, dqi.kmb and dfthcsos/."""
    if comp is None: return None
    for p in sorted(Path(comp).rglob("invsqrt.kmb")):
        root = p.parent
        if (root / "dqi.kmb").exists() and (root / "dfthcsos").is_dir(): return root
    return None

REF_ROOT = find_reference(COMP)
log("reference submission root:", REF_ROOT)
REF_FILES = sorted(p.relative_to(REF_ROOT).as_posix() for p in REF_ROOT.rglob("*") if p.is_file()) if REF_ROOT else []
print("\n".join("  " + f for f in REF_FILES))

REF = {}      # task -> loaded circuit(s)
if REF_ROOT:
    REF["invsqrt"] = km.Circuit.from_file(REF_ROOT / "invsqrt.kmb")
    REF["dqi"] = km.Circuit.from_file(REF_ROOT / "dqi.kmb")
    REF["dfthcsos"] = {p.name: km.Circuit.from_file(p) for p in sorted((REF_ROOT / "dfthcsos").glob("angle_k_*.kmb"))}

def show_reports():
    rows = []
    for t in ("invsqrt", "dqi"):
        if t in REF: rows.append((t, circuit_report(REF[t])))
    for name, c in REF.get("dfthcsos", {}).items(): rows.append((f"dfthcsos/{name}", circuit_report(c)))
    for name, r in rows:
        print(f"{name:28s} Q={r['qubits']:>6,}  instr={r['instructions']:>12,}  Toffoli={r['toffoli_like']:>10,}  "
              f"T={r['t_like']:>6,}  Z_POW(fine)={r['z_pow_fine']:>4,}  depth={r['reaction_depth']:>10,}")
        if name in ("invsqrt", "dqi"): print("   registers:", r["registers"])
    return dict(rows)
REF_REPORT = show_reports() if REF else {}

# Local check of the reference invsqrt circuit against exact arithmetic (paper §III.A.3)
INVSQRT_LOCAL = None
try:
  if "invsqrt" in REF:
    c = REF["invsqrt"]; max_x = 3 * ((1 << 10) - 2) ** 2
    target, tol = invsqrt_targets(CFG["BETA"])
    xs = (np.arange(1, max_x + 1, dtype=np.uint64) if CFG["INVSQRT_SAMPLE"] == 0
          else random_inputs(random.Random(CFG["SEED"]), 22, CFG["INVSQRT_SAMPLE"]) % np.uint64(max_x) + np.uint64(1))
    t = time.time()
    INVSQRT_LOCAL = check_phase_oracle(c, "x", xs, target, tol, seed=CFG["SEED"])
    log(f"local invsqrt check on {len(xs):,} inputs in {time.time() - t:.1f}s:", INVSQRT_LOCAL)
except Exception as e:      # e.g. the register is not called "x" in the real file: say so, keep going
    log("local invsqrt check skipped:", repr(e)); print("registers:", {str(k): len(v) for k, v in REF["invsqrt"].register_data.items()} if "invsqrt" in REF else "?")


[     1.3s] reference submission root: /kaggle/input/competitions/google-verifiable-quantum-circuit-compilation/sample_submission
  dfthcsos/angle_k_0.kmb
  dfthcsos/angle_k_1.kmb
  dfthcsos/angle_k_2.kmb
  dfthcsos/angle_k_3.kmb
  dfthcsos/angle_k_4.kmb
  dfthcsos/angle_k_5.kmb
  dfthcsos/angle_k_6.kmb
  dfthcsos/angle_k_7.kmb
  dfthcsos/angle_k_8.kmb
  dfthcsos/angle_k_9.kmb
  dfthcsos/submission.npz
  dqi.kmb
  invsqrt.kmb
invsqrt                      Q=   260  instr=      62,160  Toffoli=     7,735  T=     1  Z_POW(fine)=  37  depth=     4,005
   registers: {'x': 22}
dqi                          Q= 1,893  instr=  86,051,014  Toffoli= 5,725,461  T=     0  Z_POW(fine)=   0  depth=   170,757
   registers: {'S0': 12, 'S1': 12, 'S2': 12, 'S3': 12, 'S4': 12, 'S5': 12, 'S6': 12, 'S7': 12, 'S8': 12, 'S9': 12, 'S10': 12, 'S11': 12, 'S12': 12, 'S13': 12, 'S14': 12, 'S15': 12, 'S16': 12, 'S17': 12, 'S18': 12, 'S19': 12, 'S20': 12, 'S21': 12, 'S22': 12, 'S23': 12, 'S24': 12, 'S25': 12, 'S26': 

## 7 · Official verifiers

In [7]:
# ── 7. The official verifiers ────────────────────────────────────────────────
# Real command lines (verifiers/<task>/…), all print ONE JSON object on stdout with "status" ("PASS"/…) and "score"
# (= the task COST; lower is better; the leaderboard score is baseline_cost / cost):
#   invsqrt : verify_submission_invsqrt.py <file.kmb> [--max-x N --shots-per-x K] --salt S
#   dqi     : verify_submission_dqi.py     <file.kmb> [--num-shots N] --max-workers W --salt S
#   dfthcsos: verify.py                    <dir>      [--shots-per-address K] --salt S
# The metric appends the verifiers/ directory to sys.path and the verifiers `import kickmix`, so both go on PYTHONPATH.
SCRIPT_NAMES = {"invsqrt": "verify_submission_invsqrt.py", "dqi": "verify_submission_dqi.py", "dfthcsos": "verify.py"}
VERIFIER_FILES = {}
if COMP is not None:
    for p in sorted(Path(COMP).rglob("*")):
        rel = p.relative_to(COMP).as_posix()
        if p.is_file() and "verifier" in rel.lower(): VERIFIER_FILES[rel] = p
VERIFIER = {}
for _t, _n in SCRIPT_NAMES.items():
    _hits = [p for rel, p in VERIFIER_FILES.items() if p.name == _n and f"/{_t}/" in "/" + rel]
    if _hits: VERIFIER[_t] = _hits[0]
print(f"{len(VERIFIER_FILES)} verifier-related files | scripts found:", {t: str(p.relative_to(COMP)) for t, p in VERIFIER.items()},
      "| missing:", [t for t in SCRIPT_NAMES if t not in VERIFIER])
for rel, p in VERIFIER_FILES.items():
    if p.name.lower() == "scoring_readme.md":
        print(f"\n===== {rel} (first 3000 chars) =====\n{p.read_text(errors='replace')[:3000]}")

# Escape hatch: pin your own command per task, e.g. OFFICIAL_CMD_OVERRIDE["invsqrt"] = ["python", "x.py", "{path}", "--salt", "{salt}"]
OFFICIAL_CMD_OVERRIDE = {}

def official_cmd(task, target, salt, mode):
    if task in OFFICIAL_CMD_OVERRIDE:
        return [a.replace("{path}", str(target)).replace("{salt}", salt) for a in OFFICIAL_CMD_OVERRIDE[task]]
    cmd = [sys.executable, str(VERIFIER[task]), str(target)]
    if mode == "smoke": cmd += CFG["SMOKE_ARGS"][task]
    if task == "dqi": cmd += ["--max-workers", str(CFG["DQI_WORKERS"])]
    return cmd + ["--salt", salt]

def _official_env():
    env = dict(os.environ); paths = []
    if getattr(km, "__file__", None): paths.append(str(Path(km.__file__).parent))          # where kickmix*.so lives
    if VERIFIER: paths.append(str(next(iter(VERIFIER.values())).parent.parent))            # the verifiers/ directory
    if env.get("PYTHONPATH"): paths.append(env["PYTHONPATH"])
    env["PYTHONPATH"] = os.pathsep.join(paths)
    return env

def _parse_json_tail(text):
    for chunk in [text.strip()] + [l for l in reversed(text.strip().splitlines())]:
        try:
            obj = json.loads(chunk)
            if isinstance(obj, dict): return obj
        except Exception:
            pass
    m = re.search(r"\{.*\}", text, re.S)               # JSON possibly embedded in other output
    if m:
        try: return json.loads(m.group(0))
        except Exception: pass
    return None

def task_mode(task):
    return CFG.get("OFFICIAL_MODE_BY_TASK", {}).get(task, CFG["OFFICIAL_MODE"])

def run_official(task, path, mode=None, salt=None, timeout=None):
    """dict(ok, status, score, error, rc, seconds, cmd, tail) or None when the verifier is not available."""
    mode = mode or task_mode(task); salt = salt or CFG["OFFICIAL_SALT"]
    if not CFG["RUN_OFFICIAL"] or (task not in VERIFIER and task not in OFFICIAL_CMD_OVERRIDE): return None
    cmd = official_cmd(task, path, salt, mode); t = time.time()
    try:
        r = subprocess.run(cmd, capture_output=True, text=True, env=_official_env(), cwd=str(COMP),
                           timeout=timeout or CFG["OFFICIAL_TIMEOUT_S"])
    except subprocess.TimeoutExpired:
        return dict(ok=False, status="TIMEOUT", score=None, error="timeout", rc=None, cmd=cmd, tail="", seconds=round(time.time() - t, 1), mode=mode)
    out = (r.stdout or "") + "\n" + (r.stderr or "")
    js = _parse_json_tail(r.stdout or "")
    status = js.get("status") if js else None
    score = js.get("score") if js else None
    try: score = float(score) if score is not None else None
    except (TypeError, ValueError): score = None
    ok = (r.returncode == 0 or status == "PASS") and status == "PASS"
    return dict(ok=ok, status=status or ("NO_JSON" if r.returncode == 0 else f"EXIT_{r.returncode}"), score=score,
                error=(js or {}).get("error"), rc=r.returncode, cmd=cmd, tail=out.strip()[-2000:],
                seconds=round(time.time() - t, 1), mode=mode)

def show_failure(task, res, what=""):
    print(f"\n!! official {task} {what}: status={res['status']} rc={res['rc']} error={res.get('error')}")
    print("   command:", " ".join(map(str, res["cmd"])))
    print("   output tail:\n" + textwrap.indent(res["tail"][-1500:], "     "))

# ---- calibration: the command is only trusted if it reproduces PASS on the REFERENCE submission --------------
REF_OFFICIAL = {}
def ref_target(task): return REF_ROOT / ("dfthcsos" if task == "dfthcsos" else f"{task}.kmb")
def official_reference(task):
    key = (task, task_mode(task), CFG["OFFICIAL_SALT"])
    if key not in REF_OFFICIAL: REF_OFFICIAL[key] = run_official(task, ref_target(task))
    return REF_OFFICIAL[key]

OFFICIAL_READY = {}
if REF_ROOT:
    for _t in TASKS:
        if _t not in VERIFIER and _t not in OFFICIAL_CMD_OVERRIDE: OFFICIAL_READY[_t] = False; continue
        res = official_reference(_t)
        OFFICIAL_READY[_t] = bool(res and res["ok"])
        if OFFICIAL_READY[_t]: log(f"official {_t}: reference PASS in {res['seconds']}s ({res['mode']} mode), cost = {res['score']}")
        else: show_failure(_t, res, "on the REFERENCE (so the command is wrong, not the circuit)")
log("official verifiers calibrated:", OFFICIAL_READY, "| modes:", {t: task_mode(t) for t in TASKS})

def official_decision(task, path):
    """Run the verifier on a candidate and compare its official cost with the reference's (same mode, same salt)."""
    off = run_official(task, path); ref = official_reference(task) if REF_ROOT else None
    full = task_mode(task) == "full"
    ok = bool(off and off["ok"]); ready = bool(ref and ref["ok"])
    gain = (1 - off["score"] / ref["score"]) if (ok and ready and off["score"] is not None and ref["score"]) else None
    better = gain is not None and gain > CFG["MIN_REL_GAIN"][task]
    note = ("accepted" if (ok and ready and better and (full or CFG["ALLOW_SMOKE_SUBSTITUTION"])) else
            "verifier command not calibrated on the reference" if not ready else
            "verifier did not PASS" if not ok else
            "not cheaper than the reference" if not better else
            f"cheaper ({gain:.2%}) but only a {task_mode(task)}-mode run: set OFFICIAL_MODE_BY_TASK[{task!r}]='full' to allow shipping")
    return dict(official=off, ref_official=ref, gain=gain, accepted=note == "accepted", note=note)


18 verifier-related files | scripts found: {'invsqrt': 'verifiers/invsqrt/verify_submission_invsqrt.py', 'dqi': 'verifiers/dqi/verify_submission_dqi.py', 'dfthcsos': 'verifiers/dfthcsos/verify.py'} | missing: []

===== verifiers/dfthcsos/SCORING_README.md (first 3000 chars) =====
# `dfthcsos`: the DFTHC-SOS competition verifier

This package scores submissions to the Qaggle DFTHC-SOS competition. It is the
`verifier/` directory of the dfthc-sos repository, copied here and reduced to
the one system the competition is scored against:

| | |
| --- | --- |
| System | **FeMoCo76o113e**: `N = 76` spatial orbitals, `eta = 113` electrons |
| `E_upper` | `-22140.27335` Ha |
| Provided baseline | `R, B, C = 65, 60, 60` |
| Fixed symmetry shift | `S, beta_1` of the provided baseline, from [FeMoCo76o113e.fixed_shifts.npz](FeMoCo76o113e.fixed_shifts.npz) (SHA-256 `7b7facf63fe35a9d9d3d00787ffdb700c863fa845fe4724d7f7c54778151cd87`); submitted `H1_I` and `Hsym_lower` are ignored |
| Strict acceptance 

## 8 · Cost models

In [8]:
# ── 8. Local cost models (paper formulas) — used only to RANK candidates; the official score is the authority ──
def eps_plus(N, k=0, delta=2.0 ** -128):
    """One-sided Clopper-Pearson upper bound on the failure rate after k failures in N shots."""
    if k == 0: return 1 - delta ** (1.0 / N)
    from scipy.stats import beta as _beta
    return float(_beta.ppf(1 - delta, k + 1, N - k))

def cost_invsqrt(c, margin_turns=None):
    """S = Q * C,  C = Toffoli + T/4  (+ analytic synthesis cost of the fine Z_POW rotations)."""
    st = c.operation_statistics()
    nrot = st.num_z_pow(min_clifford_hierarchy_level=4)
    t_rot = 0.0
    if nrot:
        eps = math.pi * (margin_turns if margin_turns else 2.0 ** -26) / nrot
        t_rot = nrot * (0.56 * math.log2(1 / eps) + 5.3)
    C = st.num_toffoli_like + (st.num_t_like + t_rot) / 4
    return c.num_qubits * C

def repetition_factor(eps):
    return math.log(0.5) / math.log(0.5 + 2 * math.sqrt(eps)) if eps < 1 / 16 else math.inf

def cost_dqi(c, shots=102_400, failures=0):
    """S = R(eps+) * Q^2 * T   (single-run Toffoli count T)."""
    return repetition_factor(eps_plus(shots, failures)) * c.num_qubits ** 2 * c.operation_statistics().num_toffoli_like

def geometric_score(baseline, new):
    r = [baseline[t] / new[t] for t in TASKS if t in baseline and t in new]
    return math.exp(sum(math.log(x) for x in r) / len(r)) if r else None

if REF:
    BASE_COST = dict(invsqrt=cost_invsqrt(REF["invsqrt"]), dqi=cost_dqi(REF["dqi"]))
    log("local baseline cost proxies:", {k: f"{v:.4g}" for k, v in BASE_COST.items()},
        "| R(eps+) floor for 0/102400 failures =", f"{repetition_factor(eps_plus(102400)):.3f}")


[   525.4s] local baseline cost proxies: {'invsqrt': '2.064e+06', 'dqi': '2.444e+13'} | R(eps+) floor for 0/102400 failures = 1.191


## 9 · Optimise & gate

In [ ]:
# ── 9. Apply the exact passes to the reference circuits, verify, and keep only what is proven ──
CANDIDATES = {t: [] for t in TASKS}      # task -> list of dicts(name, path, local_cost, official, accepted)

def save_candidate(task, name, circ):
    p = WORK / "candidates" / f"{task}__{name}.kmb"
    circ.to_file(p, format="kmb"); return p

def evaluate_single(task, name, circ, ref, local_cost_fn, extra_check=None):
    """Local exactness checks -> (only if not worse by the local proxy) official verifier -> decision. Never raises."""
    row = dict(task=task, name=name, qubits=circ.num_qubits, instructions=len(circ))
    try:
        ok, why = equiv_check(ref, circ, batch=128, rounds=3)
        row.update(equiv=ok, equiv_note=why)
        if not ok: ledger(**row, accepted=False); return row
        if extra_check:
            row["extra"] = extra_check(circ)
            if not row["extra"].get("passed", False): ledger(**row, accepted=False); return row
        row["local_cost"] = local_cost_fn(circ); row["ref_cost"] = local_cost_fn(ref)
        path = save_candidate(task, name, circ); row["path"] = str(path)
        if row["local_cost"] <= row["ref_cost"]:            # official runs are expensive: skip clearly-worse candidates
            row.update(official_decision(task, path))
            if row["official"] and not row["official"]["ok"]: show_failure(task, row["official"], f"on candidate {name}")
        else:
            row.update(accepted=False, note="worse than the reference by the local cost proxy; official verifier not run")
    except Exception as e:
        row.update(error=repr(e), trace=traceback.format_exc()[-1200:], accepted=False)
    row.setdefault("accepted", False)
    ledger(**row)
    CANDIDATES[task].append(row)
    return row

if CFG["RUN_OPTIMIZER"] and REF and globals().get("SELFTEST_OK", False):
    inv_extra = (lambda c: check_phase_oracle(c, "x", xs, *invsqrt_targets(CFG["BETA"]), seed=CFG["SEED"])) if "xs" in globals() else None
    new, rep = optimize_circuit(REF["invsqrt"]); log("invsqrt passes:", rep)
    if new is not None and (rep["cancelled"] or rep["phase_merged"] or rep["phase_dropped"] or rep["qubits_reused"]):
        r = evaluate_single("invsqrt", "exact_passes", new, REF["invsqrt"], cost_invsqrt, inv_extra); log("invsqrt candidate:", {k: r.get(k) for k in ("qubits", "equiv", "local_cost", "ref_cost", "gain", "accepted", "note")})
    else:
        log("invsqrt: the exact passes find nothing to remove — the reference is already tight at this level")

    new, rep = optimize_circuit(REF["dqi"]); log("dqi passes:", rep)
    if new is not None and (rep["cancelled"] or rep["phase_merged"] or rep["phase_dropped"] or rep["qubits_reused"]):
        r = evaluate_single("dqi", "exact_passes", new, REF["dqi"], cost_dqi); log("dqi candidate:", {k: r.get(k) for k in ("qubits", "equiv", "local_cost", "ref_cost", "gain", "accepted", "note")})

    # dfthcsos: angle_k_*.kmb are verified together, so rewrite them all and verify the directory in step 10
    DFT_NEW, tot = {}, dict(cancelled=0, phase_merged=0, phase_dropped=0, qubits_reused=0)
    for name, c in REF["dfthcsos"].items():
        n2, rep = optimize_circuit(c)
        if n2 is None: continue
        ok, why = equiv_check(c, n2, batch=64, rounds=2)
        if ok and (n2.num_qubits, len(n2)) != (c.num_qubits, len(c)): DFT_NEW[name] = n2
        for k in tot: tot[k] += rep.get(k, 0)
        if not ok: log("dfthcsos", name, "REJECTED:", why)
    log(f"dfthcsos: {len(DFT_NEW)}/{len(REF['dfthcsos'])} angle circuits changed by the exact passes |", tot)


[   526.0s] invsqrt passes: {'ops_in': 62183, 'cancelled': 4222, 'phase_merged': 0, 'phase_dropped': 0, 'qubits_reused': 0, 'ops_out': 57961}


## 10 · Builders (plug-in point)

In [ ]:
# ── 10. Plug-in point for real circuit work (builders you or an agent write) ──
# A builder takes the reference circuit(s) and returns a km.Circuit (or None to skip). Whatever it returns goes through
# the SAME gate as the exact passes: circuit-vs-circuit equivalence on random inputs, the local task check (invsqrt),
# the official verifier, and a strict cost improvement.  Builders that are only approximately equal to the reference
# (shorter adders, truncated registers, different polynomial/QROM precision) cannot use the equivalence gate by design:
# register them with approximate=True and they are judged by the task check + official verifier alone.
BUILDERS = {t: [] for t in TASKS}
def register_builder(task, approximate=False):
    def deco(fn): BUILDERS[task].append((fn, approximate)); return fn
    return deco

# --- example template (kept commented; it shows the API, e.g. approximate adders via btol):
# @register_builder("invsqrt", approximate=True)
# def tighter_adders(ref):
#     b = km.CircuitBuilder(); x = b.create_quantum_register(22, name="x")
#     ... b.iadd(offset, target=acc, btol=26.0) ...      # btol < len(target): approximate carry, error <= 2**-btol of inputs
#     return b.finish_circuit()

def run_builders():
    for task in TASKS:
        for fn, approx in BUILDERS[task]:
            if time_left() < 600: log("time budget nearly spent — stopping builders"); return
            try:
                circ = fn(REF[task])
            except Exception as e:
                ledger(task=task, name=fn.__name__, error=repr(e), accepted=False); log(task, fn.__name__, "raised", repr(e)); continue
            if circ is None: continue
            if approx:
                row = dict(task=task, name=fn.__name__, qubits=circ.num_qubits, instructions=len(circ))
                if task == "invsqrt" and "xs" in globals():
                    row["extra"] = check_phase_oracle(circ, "x", xs, *invsqrt_targets(CFG["BETA"]), seed=CFG["SEED"])
                row["local_cost"] = cost_invsqrt(circ) if task == "invsqrt" else cost_dqi(circ)
                row["ref_cost"] = cost_invsqrt(REF[task]) if task == "invsqrt" else cost_dqi(REF[task])
                p = save_candidate(task, fn.__name__, circ); row["path"] = str(p)
                if row.get("extra", dict(passed=(task != "invsqrt")))["passed"]: row.update(official_decision(task, p))
                else: row.update(accepted=False, note="local task check failed")
                ledger(**row); CANDIDATES[task].append(row)
            else:
                evaluate_single(task, fn.__name__, circ, REF[task], cost_invsqrt if task == "invsqrt" else cost_dqi)

# ═════════════════════════════════════════════════════════════════════════════════════════════════
# invsqrt builders (v2 of this notebook). They reuse the official verifiers/invsqrt/nr_reference_circuit.py
# and change only what a calibrated model + exhaustive simulation showed to be safe and cheaper.
# ═════════════════════════════════════════════════════════════════════════════════════════════════
NR = None
if "invsqrt" in VERIFIER:
    _nr_path = VERIFIER["invsqrt"].parent / "nr_reference_circuit.py"
    _spec = importlib.util.spec_from_file_location("nr_reference_circuit", _nr_path)
    NR = importlib.util.module_from_spec(_spec); sys.modules["nr_reference_circuit"] = NR; _spec.loader.exec_module(NR)
    log("loaded", _nr_path)

"""Variantes del circuito invsqrt (Benchmark I) construidas sobre el constructor oficial.

Se apoya en las funciones de `verifiers/invsqrt/nr_reference_circuit.py` (módulo `nr`) y cambia
tres cosas, cada una activable por separado:

  prec          anchos de registro recortados (Precision), elegidos con el modelo de punto fijo
                oficial evaluado sobre las 3,133,452 x.
  coef_regs=3   los cuatro coeficientes de Horner usan 3 registros en vez de 4: c3 solo se
                necesita en el primer paso, y como cargarlo es Clifford (XOR) se descarga y su
                registro se reutiliza para c1. Misma aritmética, 17 qubits menos, 0 Toffolis extra.
  binary_shift  el número s de pares de ceros iniciales se guarda en binario (4 qubits) en vez de
                10 banderas unarias. Normalizar: 4 etapas (¿los 16/8/4/2 bits de arriba son cero?).
                Desnormalizar: un solo left_rotate(y1, shift=s) con desplazamiento cuántico, que
                kickmix arma con len(s)+1 inversiones en vez de 10 rotaciones.

Uso:
    circ = build_invsqrt(km, nr, prec=nr.Precision(14, 17, 28, 28, 29), coef_regs=4, binary_shift=False)
reproduce la referencia oficial compuerta por compuerta en el mismo orden de costos.
"""



def _load_coef(b, nr, prec, x, selectors, j, reg):
    """reg ^= coeficiente j del intervalo elegido por (top, t_a, t_b). Clifford y su propio inverso."""
    top, t_a, t_b = x[len(x) - 1], selectors[0], selectors[1]
    table = prec.coefficient_table()
    for control, value in ((True, table[0][j]), (top, table[0][j] ^ table[2][j]),
                           (t_a, table[0][j] ^ table[1][j]), (t_b, table[2][j] ^ table[3][j])):
        b.gf2_iadd(value, target=reg, control=control)


def _horner_step(b, nr, prec, inverse, x, selectors, acc, mult, scratch):
    """acc -= T(mult · delta)  (o += en el inverso), filas idénticas a nr._horner."""
    width, wc = len(x), len(acc)
    delta = [(selectors[2], 1)]
    delta += [(x[width - 2 - m], m) for m in range(2, min(prec.delta_bits, width - 2) + 1)]
    for control, m in delta:
        nr._add_row(b, inverse, control, [(mult[i], i - m) for i in range(m, wc)], acc, scratch, subtract=True)


def _coefs_and_horner(b, nr, prec, inverse, x, selectors, regs, scratch):
    """Carga de coeficientes + Horner. Devuelve el registro que termina con y0."""
    L = lambda j, r: _load_coef(b, nr, prec, x, selectors, j, r)
    S = lambda acc, mult: _horner_step(b, nr, prec, inverse, x, selectors, acc, mult, scratch)
    if len(regs) == 4:                       # como la referencia: c0..c3 en registros propios
        c0, c1, c2, c3 = regs
        if not inverse:
            for j, r in enumerate(regs): L(j, r)
            S(c2, c3); S(c1, c2); S(c0, c1)
        else:
            S(c0, c1); S(c1, c2); S(c2, c3)
            for j, r in enumerate(regs): L(j, r)
        return c0
    A, B, C = regs                           # A: c3 y luego c1/r2 · B: c2/r1 · C: c0/y0
    if not inverse:
        L(3, A); L(2, B); S(B, A)            # B = r1 = c2 - T(c3 d)
        L(3, A)                              # A vuelve a 0 (c3 no cambió)
        L(1, A); S(A, B)                     # A = r2 = c1 - T(r1 d)
        L(0, C); S(C, A)                     # C = y0 = c0 - T(r2 d)
    else:
        S(C, A); L(0, C)                     # C = c0 -> 0
        S(A, B); L(1, A)                     # A = c1 -> 0
        L(3, A); S(B, A); L(2, B); L(3, A)   # B = c2 -> 0, A = c3 -> 0
    return C


def _normalize_binary(b, inverse, x, s):
    """x := x · 2^(2s) con s en binario (s[j] = 1 si los 2^(j+1) bits de arriba son cero en esa etapa)."""
    width = len(x)
    stages = list(range(len(s) - 1, -1, -1))           # 3, 2, 1, 0
    for j in (reversed(stages) if inverse else stages):
        w = 2 ** (j + 1)
        top = x[width - w:]
        if inverse:
            b.right_rotate(x, w, control=s[j])
        b.x(top)
        b.multi_controlled_x(top, s[j])
        b.x(top)
        if not inverse:
            b.left_rotate(x, w, control=s[j])


def build_invsqrt(km, nr, *, prec=None, coef_regs: int = 3, binary_shift: bool = True,
                  n_p: int = 10, builder=None):
    prec = prec or nr.Precision()
    width = 2 * n_p + 2
    out_bits = prec.output_bits(n_p)
    b = builder if builder is not None else km.CircuitBuilder()

    x = b.create_quantum_register(width, name='x')
    y1 = b.alloc_qubits(out_bits)
    nflag = 4 if binary_shift else n_p
    assert (1 << nflag) > n_p
    flags = b.alloc_qubits(nflag)
    selectors = b.alloc_qubits(3)
    regs = [b.alloc_qubits(prec.coef_bits) for _ in range(coef_regs)]
    square = b.alloc_qubits(prec.square_bits)
    z = b.alloc_qubits(prec.product_bits + 2)
    scratch = b.alloc_qubits(max(prec.square_bits, prec.product_bits + 2, prec.result_bits + 1))
    p = y1[:out_bits - n_p]

    def compute(inverse):
        if not inverse:
            if binary_shift: _normalize_binary(b, False, x, flags)
            else: nr._normalize(b, False, x, flags)
            nr._select_interval(b, False, x, selectors)
            y0 = _coefs_and_horner(b, nr, prec, False, x, selectors, regs, scratch)
            nr._square(b, False, y0, square, scratch)
            nr._multiply_m(b, False, x, square, z, scratch)
            nr._correct(b, False, z)
            nr._multiply_y0(b, False, y0, z, p, scratch)
            if binary_shift: b.left_rotate(y1, flags)       # desplazamiento cuántico: y1 <<= s
            else: nr._denormalize(b, False, flags, y1)
        else:
            y0 = regs[-1] if coef_regs == 3 else regs[0]
            if binary_shift: b.right_rotate(y1, flags)
            else: nr._denormalize(b, True, flags, y1)
            nr._multiply_y0(b, True, y0, z, p, scratch)
            nr._correct(b, True, z)
            nr._multiply_m(b, True, x, square, z, scratch)
            nr._square(b, True, y0, square, scratch)
            _coefs_and_horner(b, nr, prec, True, x, selectors, regs, scratch)
            nr._select_interval(b, True, x, selectors)
            if binary_shift: _normalize_binary(b, True, x, flags)
            else: nr._normalize(b, True, x, flags)

    compute(False)
    nr._phase_by_bits(b, y1)
    compute(True)
    for reg in (y1, flags, selectors, *regs, square, z, scratch):
        b.free(reg)
    return b.finish_circuit()


INVSQRT_PREC = dict(delta_bits=12, coef_bits=17, square_bits=28, product_bits=27, result_bits=28)   # reference: 14,17,28,28,29

@register_builder("invsqrt", approximate=True)
def invsqrt_v3(ref):
    """trimmed widths + 3 coefficient registers + binary s.  Expected: Q 233, 6,935 Toffolis, ratio ~1.24."""
    if NR is None: return None
    return build_invsqrt(km, NR, prec=NR.Precision(**INVSQRT_PREC), coef_regs=3, binary_shift=True)

@register_builder("invsqrt", approximate=True)
def invsqrt_v2(ref):
    """fallback without the binary shift.  Expected: Q 239, 7,327 Toffolis, ratio ~1.15."""
    if NR is None: return None
    return build_invsqrt(km, NR, prec=NR.Precision(**INVSQRT_PREC), coef_regs=3, binary_shift=False)

run_builders() if REF else None


## 11 · Package

In [ ]:
# ── 11. Assemble, validate and zip the submission ────────────────────────────
ZIP_NAME = "submission.zip"          # check the competition's Submission page if it expects another name

def best_accepted(task):
    rows = [r for r in CANDIDATES.get(task, []) if r.get("accepted") and r.get("official") and r["official"].get("score") is not None]
    return min(rows, key=lambda r: r["official"]["score"]) if rows else None

SELECTION = {}
for t in ("invsqrt", "dqi"):
    r = best_accepted(t)
    if r: SELECTION[t] = Path(r["path"]); log(f"{t}: shipping candidate {r['name']} (official cost {r['official']['score']:.6g} vs reference {r['ref_official']['score']:.6g}, gain {r['gain']:.2%})")
    else: log(f"{t}: shipping the reference circuit")

# dfthcsos: ship the rewritten angle circuits only if the official verifier PASSES the whole directory with a lower cost
if CFG["RUN_OPTIMIZER"] and REF and globals().get("SELFTEST_OK", False) and globals().get("DFT_NEW"):
    d = WORK / "candidates" / "dfthcsos_exact"; shutil.rmtree(d, ignore_errors=True); shutil.copytree(REF_ROOT / "dfthcsos", d)
    for name, c in DFT_NEW.items(): c.to_file(d / name, format="kmb")
    tot = lambda cs: sum(c.operation_statistics().num_toffoli_like for c in cs.values())
    qmax = lambda cs: max(c.num_qubits for c in cs.values())
    merged = {n: DFT_NEW.get(n, c) for n, c in REF["dfthcsos"].items()}
    not_worse = tot(merged) <= tot(REF["dfthcsos"]) and qmax(merged) <= qmax(REF["dfthcsos"])
    dec = official_decision("dfthcsos", d)
    if dec["official"] and not dec["official"]["ok"]: show_failure("dfthcsos", dec["official"], "on the rewritten angle circuits")
    ledger(task="dfthcsos", name="exact_passes", toffoli_ref=tot(REF["dfthcsos"]), toffoli_new=tot(merged),
           qmax_ref=qmax(REF["dfthcsos"]), qmax_new=qmax(merged), **{**dec, "accepted": bool(dec["accepted"] and not_worse)})
    if dec["accepted"] and not_worse: SELECTION["dfthcsos"] = d; log("dfthcsos: shipping rewritten angle circuits")
    else: log("dfthcsos: keeping the reference —", dec["note"] if not dec["accepted"] else "a metric got worse")

def assemble(selection, out_dir):
    shutil.rmtree(out_dir, ignore_errors=True); out_dir.mkdir(parents=True)
    for rel in REF_FILES:
        src = REF_ROOT / rel
        if rel == "invsqrt.kmb" and "invsqrt" in selection: src = selection["invsqrt"]
        elif rel == "dqi.kmb" and "dqi" in selection: src = selection["dqi"]
        elif rel.startswith("dfthcsos/") and "dfthcsos" in selection:
            alt = Path(selection["dfthcsos"]) / Path(rel).name
            if alt.exists(): src = alt
        dst = out_dir / rel; dst.parent.mkdir(parents=True, exist_ok=True); shutil.copyfile(src, dst)

# ---- the file-level checks the official metric applies BEFORE it simulates anything (they are not in the standalone verifiers)
MAX_ID = 2 ** 30 // 1536                                  # qubit / bit ids must stay below 699,050
MAX_REGISTERS = {"invsqrt": 1, "dqi": 70 + 4095, "angle": 128}
MAX_NPZ_BYTES = 2 ** 26

def _n_registers(c):
    n = getattr(c, "num_registers", None)
    return n if isinstance(n, int) else len([k for k in c.register_data if isinstance(k, str)])

def format_check(root, selection):
    """Returns a list of problems (empty = the archive layout would be accepted). Never loads dqi.kmb twice."""
    root = Path(root); bad = []
    def one(pattern, what, is_dir=False):
        hits = [p for p in root.rglob(pattern) if p.is_dir() == is_dir]
        if len(hits) != 1: bad.append(f"expected exactly one {what}, found {len(hits)}"); return None
        return hits[0]
    pi, pd, dd = one("invsqrt.km[xb]", "invsqrt.kmx|kmb"), one("dqi.km[xb]", "dqi.kmx|kmb"), one("dfthcsos", "dfthcsos directory", True)
    for p in root.rglob("*"):
        st = p.lstat()
        if p.is_symlink() or (p.is_file() and st.st_nlink > 1): bad.append(f"{p.relative_to(root)} must be a regular, unlinked file")
    def check_circuit(c, what, limit):
        if c.num_qubits >= MAX_ID or c.num_bits >= MAX_ID: bad.append(f"{what}: qubit/bit ids must be < {MAX_ID:,}")
        if _n_registers(c) > limit: bad.append(f"{what}: {_n_registers(c)} registers > {limit}")
    if pi: check_circuit(km.Circuit.from_file(pi) if "invsqrt" in selection else REF["invsqrt"], "invsqrt", MAX_REGISTERS["invsqrt"])
    if pd: check_circuit(km.Circuit.from_file(pd) if "dqi" in selection else REF["dqi"], "dqi", MAX_REGISTERS["dqi"])
    if dd:
        files = list(dd.glob("angle_k_*.km[xb]")); stems = sorted(f.stem for f in files)
        if not files or stems != sorted(f"angle_k_{k}" for k in range(len(files))): bad.append("dfthcsos/ must hold angle_k_0 … angle_k_<n-1>, each once")
        for f in files:
            changed = "dfthcsos" in selection and f.name in globals().get("DFT_NEW", {})
            check_circuit(km.Circuit.from_file(f) if changed else REF["dfthcsos"][f.name], f.stem, MAX_REGISTERS["angle"])
        npz = one_npz = dd / "submission.npz"
        if not npz.exists(): bad.append("dfthcsos/submission.npz is missing")
        else:
            with zipfile.ZipFile(npz) as z:
                for n in z.namelist():
                    with z.open(n) as f: ver = np.lib.format.read_magic(f)
                    if ver != (1, 0): bad.append(f"{n}: npy format {ver}, must be 1.0")
            data = np.load(npz, allow_pickle=False)
            total = sum(data[k].size * data[k].dtype.itemsize for k in data.files)
            if total > MAX_NPZ_BYTES: bad.append(f"submission.npz holds {total:,} B > {MAX_NPZ_BYTES:,}")
            for k in data.files:
                if data[k].dtype.byteorder == ">" or data[k].dtype.kind not in "fiu": bad.append(f"npz[{k}] dtype {data[k].dtype} (must be little-endian numeric)")
    return bad

FINAL = None
if REF_ROOT:
    OUT = WORK / "submission_dir"; assemble(SELECTION, OUT)
    problems = format_check(OUT, SELECTION)
    if problems:
        log("FORMAT PROBLEMS in the assembled archive -> falling back to the pure reference:", problems)
        SELECTION.clear(); assemble(SELECTION, OUT)
        assert not format_check(OUT, SELECTION), "even the reference layout fails the format check"
    zpath = WORK / ZIP_NAME
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
        for rel in REF_FILES: z.write(OUT / rel, rel)
    with zipfile.ZipFile(zpath) as z: names = sorted(n for n in z.namelist() if not n.endswith("/"))
    assert names == sorted(REF_FILES), f"layout differs from the reference: {set(names) ^ set(REF_FILES)}"
    # end-to-end official check of what is in the zip, only for tasks whose circuit actually changed
    final_checks = {}
    for t, target in (("invsqrt", OUT / "invsqrt.kmb"), ("dqi", OUT / "dqi.kmb"), ("dfthcsos", OUT / "dfthcsos")):
        if t in SELECTION: final_checks[t] = run_official(t, target)
    manifest = dict(files={rel: sha256_file(OUT / rel) for rel in REF_FILES}, zip_sha256=sha256_file(zpath),
                    selection={k: str(v) for k, v in SELECTION.items()}, kickmix=getattr(km, "__file__", None),
                    mode=CFG["OFFICIAL_MODE"], salt=CFG["OFFICIAL_SALT"], seed=CFG["SEED"], seconds=round(elapsed(), 1),
                    official={k: (None if v is None else dict(ok=v["ok"], status=v["status"], score=v["score"])) for k, v in final_checks.items()})
    (WORK / "manifest.json").write_text(json.dumps(manifest, indent=1))
    FINAL = manifest
    log("wrote", zpath, f"({zpath.stat().st_size:,} B)  format check: OK")
    for t in TASKS:
        if t in final_checks:
            v = final_checks[t]
            print(f"  {t:9s}: CHANGED   official {v['status'] if v else 'not run'}  cost={v['score'] if v else None}")
            if v and not v["ok"]: show_failure(t, v, "on the assembled submission")
        else:
            print(f"  {t:9s}: reference (verifier {'calibrated: PASS' if OFFICIAL_READY.get(t) else 'NOT calibrated — see the output of step 7'})")
    print("\nledger rows:", len(LEDGER), "| accepted:", sum(bool(r.get('accepted')) for r in LEDGER), "| official mode:", CFG["OFFICIAL_MODE"])
else:
    log("no reference submission found — nothing to package. Attach the competition data and re-run.")


## Where the score can actually move (hypotheses from the paper — test each with the gate above)

**invsqrt** — `S = Q × (Toffoli + T/4 + rotation cost)`. Reference: 260 qubits, 7,735 Toffoli, worst relative error 2⁻²⁵·⁰⁷ against a 2⁻²⁴ tolerance, i.e. roughly one bit of precision headroom to trade for cost. Another participant's public reproducibility repo (Apache-2.0, `zjukop3/google-quantum-circuit-compilation-2026-support`) reports a ratio of 1.2648 on invsqrt by re-encoding the 10-bit unary leading-zero-pair counter as a compact binary register (Q 260 → 231, Toffoli-equivalents 7,987 → 7,108) — read it for the idea, rebuild and verify it yourself. The paper itself lists: a simpler polynomial over a wider range instead of the cubic, QROM coefficient precision, truncation widths (`T(·)`), and shifting after the Newton multiply. `CircuitBuilder.iadd(..., btol=…)` gives approximate carries when the tolerance allows.

**dqi** — `S = R(ε⁺)·Q²·T`. Zero failures in 102,400 shots floors `R` at ≈1.19, so only `Q²T` can move. Reference: 1,893 qubits, 5,725,461 Toffoli. The paper notes a 51-Toffoli GF(2¹²) multiplier (vs the 63-Toffoli Karatsuba used) from its Ref. [12], and 351 Toffoli per position for Itoh–Tsujii inversion × 4,095 positions. Because qubits are squared, a 1 % qubit cut equals a 2 % Toffoli cut.

**dfthcsos** — `S = λ_eff·T_tot·Q_tot`. Levers: batch size `J`, angle precision `brot`, tree ordering, and the `(R, B, C)` ranks with re-optimised factors under the 20 mHa Frobenius bound; floors `Λ > 49.0`, `Egap > 3.914898`. The starter-kit dataset builds a complete submission from the baseline in three steps.

Next: write builders (`@register_builder`) per lever; each one is judged by the same gate, so a wrong idea costs nothing but time.